In [ ]:
###Select folder and protein to analyze
path = 'Z:/Augusto/Images/NSPARC/20241202-NSPARC' #avoid writing a dash / at the end
protein = 'EGFP'

#path = 'D:/Augusto/Images/SoRa/23042024-SoRa' #avoid writing a dash / at the end
#protein = 'EGFP'

#Define square size
square_size = 600

#Define which channel corresponds to fluorescent protein and which one corresponds to nucleus marker ('fp' or 'nucleus')
ch1 = 'protein'
ch2 = 'beads'

In [ ]:
import os
import re
import skimage 
from skimage import io
import matplotlib.pyplot as plt
import numpy as np
import tifffile as tf
import pandas as pd
import re
import nd2reader
import pims

In [ ]:
###Iterate and save the names of the imaging files (from the same protein) to analyze

def list_protein_files(path, protein):
    # Initialize an empty list to store file names
    protein_files = []

    # Iterate over all relevant folders in the date folder
    for filename in os.listdir(path):
        # Check if the file ends with the specified suffix
        if filename.startswith(protein):
            # Add the file to the list
            protein_files.append(os.path.join(path, filename))

    return protein_files

# Example usage:
# protein_files = list_protein_files(path, protein)
# print("Files with prefix '{}':".format(protein))
# for protein_file in protein_files:
#     print(protein_file)

In [ ]:
def setDatasetChannels(image_stack, ch1, ch2):

    #check features of the nd2 file
    print("\nfeatures:\n", image_stack, "\n") #length of stack is the number of slices
    
    #Open all slices for each channel
    channel_0 = 0
    channel_1 = 1
    
    image_ch1 = []
    image_ch2 = []
    
    #Save all slices from each channel
    for Z in range(len(image_stack)):
        image_ch1.append(image_stack.get_frame_2D(c=0, t=0, z=Z)) 
        image_ch2.append(image_stack.get_frame_2D(c=1, t=0, z=Z)) 
        
    #Assign proper channels to corresponding variables for further analysis
    if ch1 == 'protein':
        image_chProtein = image_ch1
    elif ch1 == 'beads':
        image_chBeads = image_ch1
    else:
        raise ValueError("ch1 must be either 'protein' or 'beads'")

    if ch2 == 'beads':
        image_chBeads = image_ch2
    elif ch2 == 'protein':
        image_chProtein = image_ch2
    else:
        raise ValueError("ch2 must be either 'protein' or 'beads'")
        
    return(image_chProtein, image_chBeads)

# Dataset = protein_files[5]
# #Load the .nd2 file
# image_stack = pims.open(Dataset)

# image_chProtein, image_chBeads = setDatasetChannels(image_stack, ch1, ch2)


In [ ]:
#Function to select a square in the center of image
import matplotlib.patches as patches

# image_selected = image_stack[15].copy()
# square_size = 1000

def center_square(image_selected, square_size):
    
    # Obtain hight and with of original image
    image_hight = np.shape(image_selected)[0]
    image_width = np.shape(image_selected)[1]
    
    top_square = int((image_hight - square_size) / 2)
    left_square = int((image_width - square_size) / 2)
    bottom_square = int(top_square + square_size)
    right_square = int(left_square + square_size)

#     print('top_square', top_square)
#     print('left_square', left_square)
#     print('bottom_square', bottom_square)
#     print('right_square', right_square)

    mean_image_selected = np.mean(image_selected[top_square:bottom_square, left_square:right_square])

    ###plot images (keep as comment)
    #Display images
    # Create figure and axes
#     fig, ax = plt.subplots()

#     # Display the image
#     ax.imshow(image_selected)
    
#     #Create and add a rectangle patch
#     rect = patches.Rectangle((left_square, top_square), square_size, square_size, linewidth=3, edgecolor='w', facecolor='none')
#     ax.add_patch(rect)
    
#     plt.show()
#     image_selected[top_square:bottom_square, left_square:right_square] = 0
#     plt.imshow(image_selected)    
    ###
    
    return(mean_image_selected)

# mean_image_selected = center_square(image_selected, square_size)
                                                 
# print('mean_image_selected', mean_image_selected)

In [ ]:
#####Select folder, dataset, and channels
#######
#######
#This code implements functions for reading tif.ome files

#print(skimage.__version__)
#skimage reference
#Stéfan van der Walt, Johannes L. Schönberger, Juan Nunez-Iglesias, François Boulogne, Joshua D. Warner, Neil Yager, Emmanuelle Gouillart, Tony Yu and the scikit-image contributors. scikit-image: Image processing in Python. PeerJ 2:e453 (2014) https://doi.org/10.7717/peerj.453
#https://biomedicalhub.github.io/python-data/skimage.html - Read about image processing skimage

# Create an empty DataFrame with columns
date_dfe = []
protein_dfe = []
replica_dfe = []
concentration_dfe = []
dataset_dfe = []
slice_dfe = []
mean_dfe = []
square_dfe = []

# Information to plot
#Vector with mean values per slice
#Name of each concentration dataset for colors

protein_files = list_protein_files(path, protein)
print("Folders with prefix '{}':".format(protein))
for protein_file in protein_files:
    print(protein_file)
    
    #Read dataset and select channel to analyze
    image_stack = pims.open(protein_file)
    #print(len(image_stack))
    image_protein_stack, image_beads_stack = setDatasetChannels(image_stack, ch1, ch2)
    #print("shape", np.shape(image_protein_stack))
    #image_protein_shape = np.shape(image_protein_stack) 
    #print(image_protein_shape)

    for index, Zslice in enumerate(image_protein_stack):
        image_selected = Zslice
        #print("image selected", np.shape(image_selected))
        mean_image_selected = center_square(Zslice, square_size)
        #print('mean slice',  index, ':', mean_image_selected)
        
        date_e = path.split('/')[-1]
        protein_e = protein
        replica_e = re.findall(r'EGFP-brep(\d+)-', protein_file)[0]
        concentration_e = re.findall(r'-(\d+)nM', protein_file)[0]
        dataset_e = re.findall(r'_(\d+).nd2', protein_file)[0]
        slice_e = index
        mean_e = mean_image_selected
        square_e = square_size
        
#         print("date_e", date_e)
#         print("protein_e", protein_e)
#         print("replica_e", replica_e)
#         print("concentration_e", concentration_e)
#         print("dataset_e", dataset_e)
#         print("slice_e", slice_e)
#         print("mean_e", mean_e)
#         print("square_e", square_e)
        
        date_dfe.append(date_e)
        protein_dfe.append(protein_e)
        replica_dfe.append(replica_e)
        concentration_dfe.append(concentration_e)
        dataset_dfe.append(dataset_e)
        slice_dfe.append(slice_e)
        mean_dfe.append(mean_e)
        square_dfe.append(square_e)
        #save in dataframe
        #date #protein #dataset #slice #mean 
    
# Create an empty DataFrame with columns
df_protein = pd.DataFrame({"date": date_dfe, "protein": protein_dfe, "replica": replica_dfe, "concentration": concentration_dfe, "dataset": dataset_dfe, "slice": slice_dfe, "mean": mean_dfe, "square": square_dfe})   


In [ ]:
import itertools
import matplotlib.pyplot as plt

###Draw plots and extract other meaningful information
# Create a plot
plt.figure()
# Create an empty list to store legend labels
legend_labels = []

#Create empty output lists
date_output = []
protein_output = []
replica_output = []
dataset_output = []
in_4um_slices_output = []
in_4um_means_output = []
out_4um_slices_output = []
out_4um_means_output = []
concentration_nM_output = []

# Select all unique values in the 'protein', 'replica', concentration', and 'dataset' column
unique_proteins = df_protein['protein'].unique()
#print("unique_proteins:", unique_proteins)

unique_replicas = df_protein['replica'].unique()

unique_concentrations = df_protein['concentration'].unique()
#print("unique_concentrations:", unique_concentrations)

####set colors for each concentration
####
# Map unique concentrations to numerical values
unique_concentrations_numeric = unique_concentrations.astype(int)
sorted_concentrations_numeric = np.sort(unique_concentrations_numeric)
unique_concentrations_sorted = sorted_concentrations_numeric.astype(str)

concentration_mapping = {concentration: i for i, concentration in enumerate(unique_concentrations_sorted)}
# Define colors based on the number of unique concentrations
colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_concentrations_sorted)))
line_styles = ['-', '--', '-.', ':']
#print(colors)
####
####

unique_datasets = df_protein['dataset'].unique()
#print("unique_datasets:", unique_datasets)

# Get all unique combinations of protein, replica, concentration, and dataset. Columns that share all three values correspond to slices from the same dataset 
unique_proteins_replicas_concentrations_datasets = list(itertools.product(unique_proteins, unique_replicas, unique_concentrations, unique_datasets))
# Sort the list by the integer values of unique_concentrations
unique_proteins_replicas_concentrations_datasets = sorted(unique_proteins_replicas_concentrations_datasets, key=lambda x: int(x[2]), reverse=True)
#print(unique_proteins_concentrations_datasets)

# Iterate through all indexes (rows) in the DataFrame
for shared_values in unique_proteins_replicas_concentrations_datasets:
        
    print(shared_values)
    shared_values_protein, shared_values_replica, shared_values_concentration, shared_values_dataset = shared_values
    #print(shared_values_protein, shared_values_concentration, shared_values_dataset)
    
    # Boolean indexing to select rows that share all four values
    selected_df_protein_rows = df_protein[
        (df_protein['protein'] == shared_values_protein) &
        (df_protein['replica'] == shared_values_replica) &
        (df_protein['concentration'] == shared_values_concentration) &
        (df_protein['dataset'] == shared_values_dataset)]
    
    # Print selected rows
    #print(selected_df_protein_rows, "\n")
     
    #Do whatever is needed with the selected rows
    # Plot each set of shared values and store the label
    max_mean_index = selected_df_protein_rows['mean'].idxmax()
    max_mean_slice = selected_df_protein_rows.loc[max_mean_index, 'slice']
    #print(max_mean_index)
    
    if max_mean_slice < 10 or max_mean_slice > 40:
        midindex = len(selected_df_protein_rows['mean']) // 2
        max_mean_index = selected_df_protein_rows['mean'].index[midindex]
        
    in_4um_index = max_mean_index + 20 #20 slices are equivalent to 4um
    out_4um_index = max_mean_index - 20 #20 slices are equivalent to 4um

    date_o = selected_df_protein_rows.loc[in_4um_index, 'date']
    protein_o = selected_df_protein_rows.loc[in_4um_index, 'protein']
    replica_o = selected_df_protein_rows.loc[in_4um_index, 'replica']
    concentration_nM_o = selected_df_protein_rows.loc[in_4um_index, 'concentration']
    dataset_o = selected_df_protein_rows.loc[in_4um_index, 'dataset']
    in_4um_slices_o = selected_df_protein_rows.loc[in_4um_index, 'slice']
    in_4um_means_o = selected_df_protein_rows.loc[in_4um_index, 'mean']
    
    out_4um_slices_o = selected_df_protein_rows.loc[out_4um_index, 'slice']
    out_4um_means_o = selected_df_protein_rows.loc[out_4um_index, 'mean']

    date_output.append(date_o)
    protein_output.append(protein_o)
    replica_output.append(replica_o)
    concentration_nM_output.append(concentration_nM_o)
    dataset_output.append(dataset_o)
    in_4um_slices_output.append(in_4um_slices_o)
    in_4um_means_output.append(in_4um_means_o)
    out_4um_slices_output.append(out_4um_slices_o)
    out_4um_means_output.append(out_4um_means_o)
    
    # Find the index of the given string in the NumPy array
    index_concentration = np.where(unique_concentrations == shared_values_concentration)[0]
    index_replica = int(np.where(unique_replicas == shared_values_replica)[0])

    #print(index_concentration)
    plt.plot(selected_df_protein_rows['slice'], selected_df_protein_rows['mean'], color = colors[index_concentration], linestyle = line_styles[index_replica])
    legend_labels.append(f"{shared_values_protein} {shared_values_concentration} nM; Replica: {shared_values_replica}; Dataset: {shared_values_dataset}")
    
# Add labels, title, legend, and grid
plt.xlabel('Z-Slice')
plt.ylabel('Intensity (NSPARC)')
plt.title(f'Mean intensity of {protein} concentration')

#plt.xlim([10,40])
plt.ylim([0,3]) #values for mCherry-SPARKOFF
#plt.ylim([100,200]) #values for mCherry-SPARKOFF


plt.grid(True)

plt.scatter(in_4um_slices_output, in_4um_means_output, marker='o', facecolors='none', edgecolors='black', zorder=10)
plt.scatter(out_4um_slices_output, out_4um_means_output, marker='s', facecolors='none', edgecolors='black', zorder=10)

# Create a legend outside the plot area
plt.legend(legend_labels, loc='center left', bbox_to_anchor=(1, 0.5))

# Show the plot with all overlapped plots
plt.show()

df_output = pd.DataFrame({"date": date_output, "protein": protein_output, "replica": replica_output, "concentration": concentration_nM_output, "dataset": dataset_output, "slicein": in_4um_slices_output, "meanin": in_4um_means_output, "sliceout": out_4um_slices_output, "meanout": out_4um_means_output})   


In [ ]:
print(df_output)

# Specify the file path where you want to save the text file
file_path = path + '/' + path.split('/')[-1] + '-' + protein + '_NSPARCcalibration_output.txt'
print(file_path)
# Print the DataFrame to the text file
df_output.to_csv(file_path, index=False, sep='\t')  # Use '\t' as the separator for tab-separated values

print("DataFrame has been printed to", file_path)